In [ ]:
import numpy as np
m = 3
m2 = m ** 2
q = np.zeros(m2)
q[m2 // 2] = 1

In [ ]:
print(q)

[0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [ ]:
def get_P(m, p_up, p_down, p_left, p_right):
    m2 = m ** 2
    P = np.zeros((m2, m2))
    ix_map = {i + 1: (i // m, i % m) for i in range(m2)}
    for i in range(m2):
        for j in range(m2):
            r1, c1 = ix_map[i + 1]
            r2, c2 = ix_map[j + 1]
            rdiff = r1 - r2
            cdiff = c1 - c2
            if rdiff == 0:
                if cdiff == 1:
                    P[i, j] = p_left
                elif cdiff == -1:
                    P[i, j] = p_right
                elif cdiff == 0:
                    if r1 == 0:
                        P[i, j] += p_down
                    elif r1 == m - 1:
                        P[i, j] += p_up
                    if c1 == 0:
                        P[i, j] += p_left
                    elif c1 == m - 1:
                        P[i, j] += p_right
            elif rdiff == 1:
                if cdiff == 0:
                    P[i, j] = p_down
            elif rdiff == -1:
                if cdiff == 0:
                    P[i, j] = p_up
    return P

In [ ]:
P = get_P(3, 0.2, 0.3, 0.25, 0.25)

In [ ]:
print(P)

[[0.55 0.25 0.   0.2  0.   0.   0.   0.   0.  ]
 [0.25 0.3  0.25 0.   0.2  0.   0.   0.   0.  ]
 [0.   0.25 0.55 0.   0.   0.2  0.   0.   0.  ]
 [0.3  0.   0.   0.25 0.25 0.   0.2  0.   0.  ]
 [0.   0.3  0.   0.25 0.   0.25 0.   0.2  0.  ]
 [0.   0.   0.3  0.   0.25 0.25 0.   0.   0.2 ]
 [0.   0.   0.   0.3  0.   0.   0.45 0.25 0.  ]
 [0.   0.   0.   0.   0.3  0.   0.25 0.2  0.25]
 [0.   0.   0.   0.   0.   0.3  0.   0.25 0.45]]


In [ ]:
n = 10
Pn = np.linalg.matrix_power(P, n)
res = np.matmul(q, Pn)
print(np.round(res, 3))

[0.156 0.156 0.156 0.105 0.106 0.105 0.072 0.072 0.072]


In [ ]:
s = 4
n = 10 ** 6
visited = [s]

for t in range(n):
    s = np.random.choice(m2, p=P[s, :])
    visited.append(s)

In [ ]:
unique, counts = np.unique(visited, return_counts=True)
print(np.asarray((unique+1, counts)).T)

[[     1 156910]
 [     2 156854]
 [     3 158524]
 [     4 105057]
 [     5 105541]
 [     6 106575]
 [     7  69193]
 [     8  70301]
 [     9  71046]]


In [ ]:
# Markov Reward Process

In [ ]:
P = np.zeros((m2 + 1, m2 + 1)) # +1 is for extra crash state!
P[:m2, :m2] = get_P(3, 0.2, 0.3, 0.25, 0.25)
for i in range (m2):
  P[i, m2] = P[i, i] # m2 is the crash state
  P[i, i] = 0 # robot cannot stay in its state, either moves or crash
P[m2, m2] = 1 # once a robot is in crash state it stays there and cannot leave

In [ ]:
print(P)

[[0.   0.25 0.   0.2  0.   0.   0.   0.   0.   0.55]
 [0.25 0.   0.25 0.   0.2  0.   0.   0.   0.   0.3 ]
 [0.   0.25 0.   0.   0.   0.2  0.   0.   0.   0.55]
 [0.3  0.   0.   0.   0.25 0.   0.2  0.   0.   0.25]
 [0.   0.3  0.   0.25 0.   0.25 0.   0.2  0.   0.  ]
 [0.   0.   0.3  0.   0.25 0.   0.   0.   0.2  0.25]
 [0.   0.   0.   0.3  0.   0.   0.   0.25 0.   0.45]
 [0.   0.   0.   0.   0.3  0.   0.25 0.   0.25 0.2 ]
 [0.   0.   0.   0.   0.   0.3  0.   0.25 0.   0.45]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   1.  ]]


In [ ]:
n = 10 ** 5
avg_rewards = np.zeros(m2)
for s in range(m2): # for each starting state
  for i in range(n):
    crashed = False
    s_next = s
    episode_reward = 0
    while not crashed:
      s_next= np.random.choice(m2 + 1, p=P[s_next, :])
      if s_next == m2:
        crashed = True
      else:
        episode_reward += 1
    avg_rewards[s] += episode_reward
  avg_rewards[s] /= n

In [ ]:
print(np.round(avg_rewards,2))

[1.46 2.12 1.48 2.44 3.42 2.44 1.98 2.83 1.99]


In [ ]:
p_up = 0.2
p_down = 0.3
p_left = 0.25
p_right = 0.25

v_1_1 = p_up * (1 + avg_rewards[7]) + p_down * (1 + avg_rewards[2]) + p_left * (1 + avg_rewards[4]) + p_right * (1 + avg_rewards[6])

print('estimated:', np.round(v_1_1, 2))
print('actual avg.:', np.round(avg_rewards[4], 2))

estimated: 3.36
actual avg.: 3.42


In [ ]:
# Analytically Calculating State Values

In [ ]:
R = np.ones(m2 + 1) # +1 is for the extra state crash
R[-1] = 0 # crash state (last state) has reward 0

In [ ]:
gamma = 0.9999
inv = np.linalg.inv(np.eye(m2 + 1) - gamma * P) #np.eye is the identity matrix (I)
v = np.matmul(inv, np.matmul(P, R))
print(np.round(v, 2))

[1.47 2.12 1.47 2.44 3.42 2.44 1.99 2.82 1.99 0.  ]


In [ ]:
# Estimating the state values iteratively

In [ ]:
def estimate_state_values(P, m2, threshold):
  v = np.zeros(m2 + 1)  # initial state values are 0
  max_change = threshold
  terminal_state = m2
  while max_change >= threshold:
    max_change = 0
    for s in range(m2 +1):
      v_new = 0
      for s_next in range(m2 + 1):
        r = 1 * (s_next != terminal_state)
        v_new += P[s, s_next] * (r + v[s_next]) # note that we used gamma=1 here, it works in practice here
      max_change = max(max_change, np.abs(v[s] - v_new))
      v[s] = v_new
  return np.round(v, 2)

In [ ]:
estimate_state_values(P, m2, 0.01)

array([1.46, 2.11, 1.47, 2.44, 3.41, 2.44, 1.98, 2.82, 1.99, 0.  ])